# # Install dependencies

In [1]:
!pip install -q --user transformers datasets accelerate



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


# # Load training data

In [2]:
"""Load the training data.

transformers is NLP-oriented — using an inline sentiment dataset for the
demo. `data.split` shuffles rows by default (use `shuffle=False` to keep
order); `random_state` makes the split reproducible.
"""
import pandas as pd
from nubison_model import data

raw = pd.DataFrame({
    "text": [
        "I love this product", "Amazing experience", "Wonderful day",
        "Best ever made", "Excellent quality", "Pretty good overall",
        "Terrible service", "I hate it", "Awful experience",
        "Worst day ever", "Disappointing result", "Not good at all",
    ],
    "target": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

datasets = data.split(raw, {"train": 0.75, "val": 0.25}, random_state=42)
for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


train  rows=  9

val    rows=  3

# # Train (HuggingFace transformers)

In [3]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# A random-weighted tiny DistilBert (~400 KB pickled) keeps the demo's
# mlflow artifact upload small enough to fit any reasonable proxy buffer
# and avoids pulling sentencepiece. Swap to distilbert-base-uncased or
# any other checkpoint in production.
MODEL_ID = "hf-internal-testing/tiny-random-DistilBertForSequenceClassification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)


def _as_hf_dataset(X, y):
    df = X.assign(labels=y.values.ravel())
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds.map(
        lambda r: tokenizer(r["text"], padding="max_length", truncation=True, max_length=16),
        batched=True,
    )


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 88/88 [00:00<00:00, 28287.76it/s]

In [4]:
"""`train(datasets, target, *, ...)` parameters:
    datasets      — dict from `data.split` (must include "train";
                    "val" enables `t.X_val / t.y_val` + auto-scoring).
    target        — label column name(s); single string or list of strings.
    model_type    — "classifier" / "regressor" / free-form string tag.
    weights_path  — default "src/weights.pkl".
"""
from transformers import Trainer, TrainingArguments
from nubison_model import train

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    train_ds = _as_hf_dataset(t.X_train, t.y_train)
    val_ds = _as_hf_dataset(t.X_val, t.y_val)

    args = TrainingArguments(
        output_dir="/tmp/hf_trainer",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_strategy="epoch",
        logging_steps=1,
        report_to=["mlflow"],
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
    trainer.train()

    t.save(model)

print(f"run_id: {t.run_id}")


2026/05/18 11:43:05 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


2026/05/18 11:43:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


2026/05/18 11:43:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


2026/05/18 11:43:05 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


Map:   0%|          | 0/9 [00:00<?, ? examples/s]

...

/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run bouncy-fish-817 at: http://127.0.0.1:5050/#/experiments/0/runs/5340601f91d04383a149e53d2c05a8cc


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


run_id: 5340601f91d04383a149e53d2c05a8cc